# 04 - EDA: Universal Metadata (WLASL + MS-ASL)

Explores `merged_datasets/universal_metadata_has_video_wlasl_msasl_merged.csv`:
1. Frame count and duration distribution
2. FPS distribution
3. Samples per class
4. Signer vs label relationship
5. Video dimensions
6. Numeric correlations

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import chi2_contingency, spearmanr

plt.style.use('ggplot')
pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 200)

In [ ]:
PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
CSV_PATH = PROJECT_ROOT / 'merged_datasets' / 'universal_metadata_has_video_wlasl_msasl_merged.csv'

print('CSV_PATH:', CSV_PATH)
print('Exists:', CSV_PATH.exists())

df = pd.read_csv(CSV_PATH)
print('Shape:', df.shape)
df.head()

## 2. Frame Count and Duration Distribution

In [ ]:
length_stats = df['length_frames'].describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.99])
duration_stats = df['duration_sec'].describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.99])
frame_start_stats = df['start_frame'].describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.99])
print('length_frames stats:')
display(length_stats.to_frame('length_frames'))
print('duration_sec stats:')
display(duration_stats.to_frame('duration_sec'))
print('start_frame stats:')
display(frame_start_stats.to_frame('start_frame'))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

df['length_frames'].dropna().hist(bins=60, ax=axes[0], color='#1f77b4')
axes[0].set_title('Frame Count (length_frames)')
axes[0].set_xlabel('Frames')
axes[0].set_ylabel('Count')

df['duration_sec'].dropna().hist(bins=60, ax=axes[1], color='#ff7f0e')
axes[1].set_title('Sign Duration (seconds)')
axes[1].set_xlabel('Duration [s]')
axes[1].set_ylabel('Count')

plt.tight_layout()
plt.show()

## 3. FPS Distribution

In [ ]:
fps_stats = df['fps'].describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.99])
display(fps_stats.to_frame('fps'))

fps_counts = df['fps'].round(3).value_counts().sort_index()
print('Most common FPS values:')
display(fps_counts.sort_values(ascending=False).head(15).to_frame('count'))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

df['fps'].dropna().hist(bins=50, ax=axes[0], color='#9467bd')
axes[0].set_title('FPS Histogram')
axes[0].set_xlabel('FPS')
axes[0].set_ylabel('Count')

axes[1].boxplot(df['fps'].dropna(), vert=False)
axes[1].set_title('FPS Boxplot')
axes[1].set_xlabel('FPS')

plt.tight_layout()
plt.show()

## 4. Samples per Class

In [ ]:
label_counts = df['label'].value_counts()
print('Unique classes:', df['label'].nunique())
print('Mean samples/class:', round(label_counts.mean(), 2))
print('Median samples/class:', round(label_counts.median(), 2))
print('Min/Max samples/class:', int(label_counts.min()), '/', int(label_counts.max()))

display(label_counts.describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.99]).to_frame('samples_per_label'))
display(label_counts.head(20).to_frame('count').rename_axis('label'))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

label_counts.hist(bins=50, ax=axes[0], color='#17becf')
axes[0].set_title('Samples per Class (histogram)')
axes[0].set_xlabel('Samples per class')
axes[0].set_ylabel('Number of classes')

label_counts.sort_values(ascending=False).head(30).plot(kind='bar', ax=axes[1], color='#8c564b')
axes[1].set_title('Top 30 Classes by Sample Count')
axes[1].set_xlabel('Label')
axes[1].set_ylabel('Samples')

plt.tight_layout()
plt.show()

## 5. Signer vs Label

In [ ]:
signer_counts = df['signer_id'].value_counts(dropna=False)
labels_per_signer = df.groupby('signer_id')['label'].nunique().sort_values(ascending=False)

print('Unique signers:', df['signer_id'].nunique())
print('Mean samples/signer:', round(signer_counts.mean(), 2))
print('Mean unique labels/signer:', round(labels_per_signer.mean(), 2))

display(signer_counts.head(20).to_frame('samples').rename_axis('signer_id'))
display(labels_per_signer.head(20).to_frame('unique_labels').rename_axis('signer_id'))

In [ ]:
top_signers = signer_counts.head(20).index
top_labels = label_counts.head(20).index

heat = pd.crosstab(df['signer_id'], df['label']).reindex(index=top_signers, columns=top_labels, fill_value=0)

fig, ax = plt.subplots(figsize=(12, 8))
im = ax.imshow(heat.values, aspect='auto')
ax.set_title('Signer vs Label (Top 20 x Top 20)')
ax.set_xlabel('Label')
ax.set_ylabel('Signer ID')
ax.set_xticks(np.arange(len(heat.columns)))
ax.set_xticklabels(heat.columns, rotation=90)
ax.set_yticks(np.arange(len(heat.index)))
ax.set_yticklabels(heat.index)
fig.colorbar(im, ax=ax, label='Samples')
plt.tight_layout()
plt.show()

In [ ]:
contingency = pd.crosstab(df['signer_id'], df['label'])
chi2, p_value, dof, _ = chi2_contingency(contingency)
n = contingency.values.sum()
r, k = contingency.shape
cramers_v = np.sqrt((chi2 / n) / max(min(r - 1, k - 1), 1))

print(f'Chi2: {chi2:.2f}')
print(f'p-value: {p_value:.6e}')
print(f"Cramer's V: {cramers_v:.4f}")

if cramers_v < 0.1:
    print('Interpretation: very weak association')
elif cramers_v < 0.3:
    print('Interpretation: weak/moderate association')
elif cramers_v < 0.5:
    print('Interpretation: moderate/strong association')
else:
    print('Interpretation: strong association')

## 6. Video Dimensions

In [ ]:
size_df = df[['video_width', 'video_height']].dropna()
print('Samples with known dimensions:', len(size_df))
display(size_df.describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.99]))

top_res = df.dropna(subset=['video_width', 'video_height']).assign(res=lambda x: x['video_width'].astype(int).astype(str) + 'x' + x['video_height'].astype(int).astype(str))['res'].value_counts().head(15)
print('Most common resolutions:')
display(top_res.to_frame('count'))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 4))

df['video_width'].dropna().hist(bins=50, ax=axes[0], color='#1f77b4')
axes[0].set_title('Video Width')
axes[0].set_xlabel('Width [px]')
axes[0].set_ylabel('Count')

df['video_height'].dropna().hist(bins=50, ax=axes[1], color='#ff7f0e')
axes[1].set_title('Video Height')
axes[1].set_xlabel('Height [px]')
axes[1].set_ylabel('Count')

plt.tight_layout()
plt.show()

In [ ]:
res_df = df.dropna(subset=['video_width', 'video_height']).copy()
res_df = res_df[(res_df['video_width'] > 0) & (res_df['video_height'] > 0)]
res_df['resolution'] = res_df['video_width'].astype(int).astype(str) + 'x' + res_df['video_height'].astype(int).astype(str)

preset_order = ['640x480', '1280x720', '1920x1080', '320x240', '854x480']
preset_counts = res_df['resolution'].value_counts()

preset_plot = pd.Series({preset: int(preset_counts.get(preset, 0)) for preset in preset_order})
other_count = int(len(res_df) - preset_plot.sum())
preset_plot['other'] = other_count

fig, ax = plt.subplots(figsize=(10, 4))
preset_plot.plot(kind='bar', ax=ax, color=['#1f77b4', '#ff7f0e', '#2ca02c', '#9467bd', '#8c564b', '#7f7f7f'])
ax.set_title('Resolution Presets')
ax.set_xlabel('Preset')
ax.set_ylabel('Count')

for i, val in enumerate(preset_plot.values):
    ax.text(i, val + max(1, 0.01 * max(preset_plot.values)), str(val), ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

print('Top 15 resolutions:')
display(preset_counts.head(15).to_frame('count'))

In [ ]:
sample_scatter = df.dropna(subset=['video_width', 'video_height']).sample(min(3000, df.dropna(subset=['video_width', 'video_height']).shape[0]), random_state=42)

fig, ax = plt.subplots(figsize=(7, 6))
ax.scatter(sample_scatter['video_width'], sample_scatter['video_height'], alpha=0.25, s=10)
ax.set_title('Width vs Height')
ax.set_xlabel('Width [px]')
ax.set_ylabel('Height [px]')
plt.tight_layout()
plt.show()

## 7. Numeric Correlations

In [ ]:
num_cols = ['start_frame', 'end_frame', 'length_frames', 'fps', 'video_width', 'video_height', 'duration_sec']
num_df = df[num_cols].copy()

pearson_corr = num_df.corr(method='pearson')
spearman_corr = num_df.corr(method='spearman')

print('Pearson correlation matrix:')
display(pearson_corr.round(3))
print('Spearman correlation matrix:')
display(spearman_corr.round(3))

In [ ]:
corr = pearson_corr.fillna(0).values
labels = pearson_corr.columns.tolist()

fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(corr, vmin=-1, vmax=1, cmap='coolwarm')
ax.set_xticks(np.arange(len(labels)))
ax.set_yticks(np.arange(len(labels)))
ax.set_xticklabels(labels, rotation=90)
ax.set_yticklabels(labels)
ax.set_title('Pearson Correlation Heatmap')
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
plt.tight_layout()
plt.show()

In [ ]:
source_avail = pd.crosstab(df['source'], df['has_video'], normalize='index') * 100
source_counts = df['source'].value_counts()

display(source_counts.to_frame('samples'))
display(source_avail.round(2))

if True in source_avail.columns:
    top_have = source_avail[True].sort_values(ascending=False).head(10)
    fig, ax = plt.subplots(figsize=(9, 4))
    top_have.plot(kind='bar', ax=ax, color='#2ca02c')
    ax.set_title('Top Sources by Video Availability')
    ax.set_ylabel('% available')
    ax.set_xlabel('source')
    plt.tight_layout()
    plt.show()

## 8. Label vs Sign Duration

In [ ]:
base_df = df_existing.copy() if 'df_existing' in globals() else df[df['has_video'] == True].copy()
label_duration_df = base_df.dropna(subset=['label', 'duration_sec']).copy()
label_length_df = base_df.dropna(subset=['label', 'length_frames']).copy()

label_duration_stats = label_duration_df.groupby('label').agg(
    samples=('duration_sec', 'size'),
    mean_duration=('duration_sec', 'mean'),
    median_duration=('duration_sec', 'median'),
    std_duration=('duration_sec', 'std'),
    q25_duration=('duration_sec', lambda s: s.quantile(0.25)),
    q75_duration=('duration_sec', lambda s: s.quantile(0.75)),
).sort_values('samples', ascending=False)

label_length_stats = label_length_df.groupby('label').agg(
    samples=('length_frames', 'size'),
    mean_frames=('length_frames', 'mean'),
    median_frames=('length_frames', 'median'),
    std_frames=('length_frames', 'std'),
).sort_values('samples', ascending=False)

print('has_video=True rows only')
print('Top 20 classes: duration stats [seconds]')
display(label_duration_stats.head(20).round(3))

print('Top 20 classes: duration stats [frames]')
display(label_length_stats.head(20).round(2))

min_samples = 10
duration_corr_df = label_duration_stats[label_duration_stats['samples'] >= min_samples].copy()

if len(duration_corr_df) >= 3:
    rho, p_val = spearmanr(duration_corr_df['samples'], duration_corr_df['median_duration'])
    print(f'Spearman(samples, median_duration) for classes with >= {min_samples} samples: rho={rho:.4f}, p-value={p_val:.4e}')
else:
    print('Too few classes after filtering for Spearman correlation.')

In [ ]:
top_labels_by_samples = label_duration_stats.head(15).index
plot_df = label_duration_df[label_duration_df['label'].isin(top_labels_by_samples)].copy()

fig, axes = plt.subplots(1, 2, figsize=(18, 5))

plot_df.boxplot(column='duration_sec', by='label', ax=axes[0], rot=90)
axes[0].set_title('Sign Duration [s] per Label (Top 15)')
axes[0].set_xlabel('Label')
axes[0].set_ylabel('Duration [s]')

axes[1].scatter(
    duration_corr_df['samples'],
    duration_corr_df['median_duration'],
    alpha=0.75,
    s=35,
    color='#1f77b4'
)
axes[1].set_title('Sample Count vs Median Duration')
axes[1].set_xlabel('Samples in class')
axes[1].set_ylabel('Median duration [s]')

if len(duration_corr_df) >= 2:
    m, b = np.polyfit(duration_corr_df['samples'], duration_corr_df['median_duration'], 1)
    x = np.linspace(duration_corr_df['samples'].min(), duration_corr_df['samples'].max(), 100)
    axes[1].plot(x, m * x + b, color='#d62728', linewidth=2, label='linear trend')
    axes[1].legend()

plt.suptitle('')
plt.tight_layout()
plt.show()